# Qwen3-14B Inference API (Kaggle Notebook)

This notebook runs the **Qwen3-14B** language model on **Kaggle's free GPU** and exposes it as a small **HTTP API** that the local Streamlit app (`app.py`) talks to.

**Pipeline**

    Kaggle GPU  ->  load Qwen3-14B  ->  FastAPI (POST /generate)  ->  ngrok tunnel  ->  public API URL

**What the local app sends**

    POST <KAGGLE_API_URL>/generate
    Authorization: Bearer <API_SECRET>
    {"prompt": "Summarize this document...", "max_new_tokens": 500}

**What it gets back**

    {"response": "..."}

**Before you run**

1. Turn on the GPU: **Settings -> Accelerator -> GPU**.
2. Recommended: add your free ngrok token as a Kaggle Secret named `NGROK_AUTHTOKEN` (free token at https://dashboard.ngrok.com/get-started/your-authtoken).
3. Optional: add a Kaggle Secret named `API_SECRET` to pick your own password; otherwise one is generated for you.
4. Run the cells from top to bottom.

## Step 1 - Install the missing packages

Kaggle already ships PyTorch, Transformers and Hugging Face, but not the web-server and tunnel packages. This cell installs only what is missing (`fastapi`, `uvicorn`, `pyngrok`, `accelerate`) and does nothing if they are already present.

In [ ]:
import importlib.util
import os
import secrets
import subprocess
import sys
import threading

# Packages Kaggle does not ship with.
REQUIRED_PACKAGES = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "pyngrok": "pyngrok",
    "accelerate": "accelerate",
}


def install_missing_packages() -> None:
    """pip install anything that is missing (silent when already present)."""
    missing = [
        pip_name
        for pip_name, module in REQUIRED_PACKAGES.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing:
        print(f"Installing missing packages: {', '.join(missing)}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *missing]
        )


install_missing_packages()

## Step 2 - Load secrets and read the configuration

**First cell** reads the Kaggle Secrets you added (Settings -> Secrets) into environment variables. Everything is optional, so it is safe if you added none.

**Second cell** imports the libraries and defines the settings:

| Setting | What it does |
| --- | --- |
| `MODEL_NAME` | The full 14B model (needs ~28 GB of VRAM). |
| `QUANTIZED_MODEL_NAME` | The 4-bit AWQ version, chosen automatically when the GPU has less than 34 GB (Kaggle's T4 has 16 GB). |
| `PORT` | The local port the API listens on (default `8000`). |
| `TEMPERATURE` / `TOP_P` | Sampling settings, i.e. how creative the answers are. |
| `API_SECRET` | The password for the API, taken from a Kaggle Secret or randomly generated so you can copy it into your local `.env`. |

In [ ]:
# Read optional Kaggle Secrets into the environment (Settings -> Secrets).
# Add "NGROK_AUTHTOKEN" (recommended) and optionally "API_SECRET".
try:
    from kaggle_secrets import UserSecretsClient

    _client = UserSecretsClient()
    for _name in ("NGROK_AUTHTOKEN", "API_SECRET"):
        try:
            os.environ[_name] = _client.get_secret(_name)
            print(f"Loaded secret: {_name}")
        except Exception:
            print(f"No secret named {_name} (optional).")
except ImportError:
    print("Not running on Kaggle - skipping Kaggle secrets.")

In [ ]:
import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Model configuration ----------------------------------------------------
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-14B")

# 14B weights need ~28 GB in fp16/bf16. On smaller GPUs we load the official
# 4-bit AWQ build instead; architecture and prompts stay exactly the same.
QUANTIZED_MODEL_NAME = os.getenv("QUANTIZED_MODEL_NAME", "Qwen/Qwen3-14B-AWQ")
FULL_MODEL_MIN_MEMORY_GB = 34

# --- Server configuration ---------------------------------------------------
PORT = int(os.getenv("PORT", "8000"))
TEMPERATURE = 0.7
TOP_P = 0.9

# Secret shared with the local app. Read from a Kaggle Secret above, or
# generated here so you can copy it into your local .env file.
API_SECRET = os.getenv("API_SECRET", "").strip() or secrets.token_urlsafe(24)

tokenizer = None
model = None
_generation_lock = threading.Lock()

## Step 3 - Load the model

The code makes three decisions automatically:

- **Which model** - if the GPU has less than 34 GB of memory, load the 4-bit AWQ build (same model, roughly 4x smaller).
- **Which precision** - `bfloat16` on modern GPUs (Ampere or newer), `float16` on older GPUs, `float32` on CPU.
- **Where the weights live** - `device_map="auto"` spreads them over the available GPU(s).

The first run downloads the weights (a few minutes). The cell prints `Model ready.` when the model is loaded.

In [ ]:
def gpu_memory_gb() -> float:
    """Total GPU memory in GB (0.0 when CUDA is not available)."""
    if not torch.cuda.is_available():
        return 0.0
    total = sum(
        torch.cuda.get_device_properties(i).total_memory
        for i in range(torch.cuda.device_count())
    )
    return total / 1e9


def select_model_name() -> str:
    """Pick the full or the 4-bit model depending on available GPU memory."""
    memory = gpu_memory_gb()
    if memory and memory < FULL_MODEL_MIN_MEMORY_GB:
        print(
            f"GPU memory {memory:.0f} GB < {FULL_MODEL_MIN_MEMORY_GB} GB -> "
            f"using {QUANTIZED_MODEL_NAME}"
        )
        return QUANTIZED_MODEL_NAME
    return MODEL_NAME


def select_dtype():
    """bf16 on Ampere+ GPUs, fp16 on older GPUs, fp32 on CPU."""
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16


def load_model() -> None:
    """Load Qwen3-14B (or its 4-bit version) onto the GPU."""
    global tokenizer, model

    name = select_model_name()
    dtype = select_dtype()
    print(f"Loading {name} (dtype={dtype}) ...")

    tokenizer = AutoTokenizer.from_pretrained(name)

    kwargs = dict(device_map="auto", low_cpu_mem_usage=True)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kwargs)
    except TypeError:
        # Older transformers versions call the argument "torch_dtype".
        model = AutoModelForCausalLM.from_pretrained(
            name, torch_dtype=dtype, **kwargs
        )
    except Exception as exc:
        if "awq" in name.lower():
            raise RuntimeError(
                "Could not load the quantized model. Install the AWQ backend "
                "first with: pip install autoawq"
            ) from exc
        raise

    if torch.cuda.is_available():
        print(f"CUDA available: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory: {gpu_memory_gb():.1f} GB total across "
              f"{torch.cuda.device_count()} device(s)")
    else:
        print("WARNING: CUDA not available, running on CPU (this will be slow).")
    print("Model ready.")

In [ ]:
# Downloads the weights on the first run, then loads them onto the GPU.
load_model()

## Step 4 - Text generation

- `build_prompt()` wraps your text in Qwen's official chat format with **thinking mode switched off**, so the model answers directly instead of writing a long hidden reasoning trace.
- `generate_text()` runs the model with `temperature=0.7` and `top_p=0.9` and returns **only the newly generated text** (the prompt is removed from the output).

In [ ]:
def build_prompt(prompt: str) -> str:
    """Wrap the raw prompt in Qwen's chat template (thinking mode disabled)."""
    messages = [{"role": "user", 
                 "content": prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        # Template does not accept enable_thinking; use its default.
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def generate_text(prompt: str, max_new_tokens: int) -> str:
    """Run Qwen3 and return only the newly generated text."""
    input_ids = tokenizer(build_prompt(prompt), return_tensors="pt").input_ids
    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Step 5 - Quick test (optional, recommended)

Generate one short answer by hand before starting the API. If a sensible sentence appears, the model is loaded and answering correctly.

In [ ]:
# Optional smoke test: one short generation to confirm everything works.
test_prompt = "In one sentence, what is a retrieval-augmented generation (RAG) pipeline?"
print(generate_text(test_prompt, max_new_tokens=120))

## Step 6 - The FastAPI endpoint

`POST /generate` takes a JSON body (`prompt`, `max_new_tokens`) and answers with `{"response": "..."}`.

Checks performed on every request:

| Result | Meaning |
| --- | --- |
| `401` | Wrong or missing `Authorization: Bearer <API_SECRET>` header. |
| `503` | The model is still loading. |
| `500` | Generation failed (only the error type is returned, never a stack trace). |
| `200` | Success - the generated text. |

A lock guarantees that only **one request uses the GPU at a time**, so parallel calls wait in line instead of crashing.

In [ ]:
class GenerateRequest(BaseModel):
    prompt: str = Field(..., min_length=1, description="Prompt to send to Qwen3")
    max_new_tokens: int = Field(default=500, ge=1, le=4096)


class GenerateResponse(BaseModel):
    response: str


app = FastAPI(title="Qwen3-14B API")


@app.post("/generate", response_model=GenerateResponse)
def generate(
    request: GenerateRequest,
    authorization: str = Header(default=""),
) -> GenerateResponse:
    if authorization != f"Bearer {API_SECRET}":
        raise HTTPException(
            status_code=401,
            detail="Invalid or missing API secret.",
        )
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model is not loaded yet.")

    with _generation_lock:
        try:
            text = generate_text(request.prompt, request.max_new_tokens)
        except Exception as exc:
            raise HTTPException(
                status_code=500,
                detail=f"Generation failed: {type(exc).__name__}",
            ) from exc

    return GenerateResponse(response=text)

## Step 7 - Open the public tunnel and start the API

1. `start_tunnel()` exposes the local port through **ngrok** and returns a public `https://...` URL.
2. The cell prints the two values you must copy into your local `.env` file: `KAGGLE_API_URL=...` and `API_SECRET=...`.
3. `uvicorn` starts in a **background thread**, so the cell finishes right away and the notebook stays usable.

**Important:** keep this notebook running while you use the local app - stopping the kernel shuts the API down. Run this cell once; run it again only after restarting the session.

In [ ]:
def start_tunnel(port: int) -> str | None:
    """Open a public ngrok tunnel and return its URL (None on failure)."""
    try:
        from pyngrok import ngrok

        auth_token = os.getenv("NGROK_AUTHTOKEN", "").strip()
        if auth_token:
            ngrok.set_auth_token(auth_token)

        tunnel = ngrok.connect(port, "http")
        return tunnel.public_url
    except Exception as exc:
        print(f"WARNING: could not start the ngrok tunnel ({exc}).")
        print(
            "Set the NGROK_AUTHTOKEN Kaggle secret (free token from "
            "https://dashboard.ngrok.com/get-started/your-authtoken) and rerun."
        )
        return None


# 1. Open the public tunnel.
public_url = start_tunnel(PORT)

# 2. Print the two values for the local .env file.
print()
print("=" * 64)
print("Copy these two values into your local .env file:")
if public_url:
    print(f"KAGGLE_API_URL={public_url}")
else:
    print(f"KAGGLE_API_URL=(ngrok failed, local url was http://127.0.0.1:{PORT})")
print(f"API_SECRET={API_SECRET}")
print("=" * 64)
print()

# 3. Start the API in a background thread so this cell finishes immediately
#    and the notebook keeps working while the API serves requests.
import uvicorn

threading.Thread(
    target=uvicorn.run,
    kwargs={"app": app, "host": "0.0.0.0", "port": PORT},
    daemon=True,
).start()
print(f"API server is running on port {PORT}. Leave this notebook running.")

## Done - connect the local app

1. Paste the printed `KAGGLE_API_URL` and `API_SECRET` into the `.env` file of your project.
2. On your computer run:

       streamlit run app.py

3. Upload a PDF and ask questions - every answer is produced by this notebook's model.

**Troubleshooting**

- **No URL printed / ngrok fails** -> add the `NGROK_AUTHTOKEN` Kaggle Secret and rerun Step 7.
- **`Could not load the quantized model`** -> run `!pip install autoawq` in a new cell, then rerun Steps 3 to 7.
- **`401 Invalid or missing API secret`** -> the `.env` value does not match the printed one; copy it again.
- **Session restarted** -> rerun every cell from the top.